# TradeFlowGNN: Multi-Model Evaluation & Comparison

This notebook provides a comprehensive benchmark across 5 models:
1. **Gravity Model (OLS)**: Traditional econometric baseline.
2. **MLP Baseline**: Feed-forward network (tabular).
3. **XGBoost Baseline**: Gradient Boosted Trees (tabular).
4. **TradeFlow GCN**: Graph Convolutional Network.
5. **TradeFlow GAT**: Graph Attention Network.

In [ ]:
# ── Colab Managed Setup (Hosted Runtime Only) ──────────────────────
import sys
from pathlib import Path
import os

def setup_colab(repo_url="https://github.com/tolyaho/TradeFlowGCN.git", branch="main"):
    if 'google.colab' in str(get_ipython()):
        if Path("/content").exists() and not Path("src").exists():
            print(f"Running in Hosted Colab. Cloning {branch} branch...")
            !pip install -q torch-geometric pytorch-lightning xgboost pyarrow pandas numpy matplotlib seaborn tqdm pyyaml scikit-learn
            repo_name = "TradeFlowGCN"
            if not Path(repo_name).exists():
                !git clone -b {branch} {repo_url}
            os.chdir(repo_name)
            
        if Path(".git").exists():
            print("Pulling latest code...")
            !git pull
        
        if os.getcwd() not in sys.path: sys.path.append(os.getcwd())
        src_path = os.path.join(os.getcwd(), "src")
        if src_path not in sys.path: sys.path.append(src_path)
    else:
        print("Running in Local Runtime. No cloning needed.")

setup_colab(branch="main")

# ── Robust Path Setup ──────────────────────────────────────────────
curr_path = Path(".").resolve()
root_path = curr_path
while not (root_path / "src").exists() and root_path.parent != root_path:
    root_path = root_path.parent

if (root_path / "src").exists():
    if str(root_path / "src") not in sys.path: sys.path.append(str(root_path / "src"))
    print(f"Project Root: {root_path}")
else:
    print(f"Warning: Could not find 'src' directory relative to {curr_path}")

## 0. Training GNNs & DL Baselines
Run these to generate checkpoints. These take ~1-2 mins each on GPU.

In [ ]:
!python scripts/train.py --config configs/default.yaml --model gcn
!python scripts/train.py --config configs/default.yaml --model gat
!python scripts/train.py --config configs/default.yaml --model mlp_baseline

## 1. Load Data & Trained Models

In [ ]:
import torch
import numpy as np
import pandas as pd
from trade_flow_gcn.utils.config import load_config
from trade_flow_gcn.data.preprocessing import preprocess_pipeline
from trade_flow_gcn.data.dataset import build_graphs_from_dataframe, TradeDataModule
from trade_flow_gcn.models.gcn import TradeFlowGCN
from trade_flow_gcn.models.gat import TradeFlowGAT
from trade_flow_gcn.models.mlp_baseline import MLPBaseline
from trade_flow_gcn.training.lightning_module import TradeFlowModule

# Load config
config = load_config(str(root_path / "configs/default.yaml"))

# Load data
raw_dir = root_path / config['data']['raw_dir']
csv_candidates = list(raw_dir.glob("*.csv"))
csv_path = max(csv_candidates, key=lambda p: p.stat().st_size)

df = preprocess_pipeline(csv_path, config)
graphs = build_graphs_from_dataframe(df, config['data']['countries'], config)

dm = TradeDataModule(
    graphs=graphs,
    train_years=tuple(config['data']['train_years']),
    val_years=tuple(config['data']['val_years']),
    test_years=tuple(config['data']['test_years'])
)
dm.setup()

def load_model_from_logs(model_type, model_class, base_dir):
    ckpt_files = list(base_dir.glob(f"{model_type}/**/checkpoints/*.ckpt"))
    if not ckpt_files: return None
        
    latest_ckpt = sorted(ckpt_files, key=lambda p: p.stat().st_mtime)[-1]
    
    if model_type == "gcn":
        net = model_class(node_input_dim=len(config['data']['node_features']), edge_input_dim=len(config['data']['edge_features']))
    elif model_type == "gat":
        net = model_class(node_input_dim=len(config['data']['node_features']), edge_input_dim=len(config['data']['edge_features']), hidden_dim=32, heads=4)
    else:
        net = model_class(input_dim=len(config['data']['node_features'])*2 + len(config['data']['edge_features']))
        
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        module = TradeFlowModule.load_from_checkpoint(latest_ckpt, model=net).to(device)
        module.eval()
        print(f"✅ Loaded {model_type} to {device}")
        return module
    except Exception as e:
        print(f"❌ Failed to load {model_type}: {e}")
        return None

log_dir = root_path / "lightning_logs"
gcn_module = load_model_from_logs("gcn", TradeFlowGCN, log_dir)
gat_module = load_model_from_logs("gat", TradeFlowGAT, log_dir)
mlp_module = load_model_from_logs("mlp_baseline", MLPBaseline, log_dir)

## 2. Tabular Baselines (Gravity & XGBoost)

In [ ]:
from trade_flow_gcn.models.gravity_baseline import GravityBaseline
from trade_flow_gcn.models.xgboost_baseline import XGBoostBaseline

def extract_numpy(data_list):
    X_src, X_dst, E_attr, Y = [], [], [], []
    for graph in data_list:
        src, dst = graph.edge_index
        X_src.append(graph.x[src].numpy())
        X_dst.append(graph.x[dst].numpy())
        E_attr.append(graph.edge_attr.numpy())
        Y.append(graph.y.numpy())
    return (np.concatenate(X_src), np.concatenate(X_dst), np.concatenate(E_attr), np.concatenate(Y))

x_s_train, x_d_train, e_train, y_train = extract_numpy(dm.train_graphs)
x_s_val, x_d_val, e_val, y_val = extract_numpy(dm.val_graphs)
x_s_test, x_d_test, e_test, y_test = extract_numpy(dm.test_graphs)

# Gravity OLS
gravity = GravityBaseline()
gravity.fit(x_s_train, x_d_train, e_train, y_train)
gravity_metrics = gravity.evaluate(x_s_test, x_d_test, e_test, y_test)

# XGBoost
xgb_val_X = np.concatenate([x_s_val, x_d_val, e_val], axis=1)
xgb_model = XGBoostBaseline()
xgb_model.fit(x_s_train, x_d_train, e_train, y_train, eval_set=[(xgb_val_X, y_val)])
xgb_metrics = xgb_model.evaluate(x_s_test, x_d_test, e_test, y_test)

print("✅ Tabular baselines fitted.")

## 3. Final Comparison Summary

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch_geometric

def evaluate_torch_module(module, graphs):
    device = next(module.parameters()).device
    loader = torch_geometric.loader.DataLoader(graphs, batch_size=1)
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = module(batch)
            preds.append(out.cpu().numpy())
            targets.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    return {
        "rmse": float(np.sqrt(mean_squared_error(targets, preds))),
        "mae": float(mean_absolute_error(targets, preds)),
        "r2": float(r2_score(targets, preds))
    }

results = []
results.append({"Model": "Gravity (OLS)", **gravity_metrics})
results.append({"Model": "XGBoost", **xgb_metrics})

if mlp_module: results.append({"Model": "MLP Baseline", **evaluate_torch_module(mlp_module, dm.test_graphs)})
if gcn_module: results.append({"Model": "TradeFlow GCN", **evaluate_torch_module(gcn_module, dm.test_graphs)})
if gat_module: results.append({"Model": "TradeFlow GAT", **evaluate_torch_module(gat_module, dm.test_graphs)})

summary_df = pd.DataFrame(results).sort_values("rmse")
display(summary_df)